# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Sinh viên:** Hoàng Quốc Việt — MSSV 2A202602563.

Notebook này điều phối toàn bộ bài lab; phần ruột nằm trong các file `.py` cùng thư mục
(`dataset.py`, `model.py`, `losses.py`, `train.py`, `inference.py`, `benchmark.py`, `run_experiments.py`,
`inference_experiments.py`, `final_eval.py`). Mọi thí nghiệm huấn luyện đi qua **một** hàm `train.run(Config(...))`.

**Chạy trên Kaggle (GPU T4 x2).** Vì mỗi bước phải chốt lựa chọn trên **val** trước khi sang bước sau,
notebook được chạy thành 4 phiên Kaggle, chọn phần chạy bằng biến môi trường `LAB_STAGE`:

| `LAB_STAGE` | Nội dung | Ghi chú |
|---|---|---|
| `A` | Bước 0 (kiểm tra chia dữ liệu, EDA, kiểm tra pipeline) + Bước 1 (7 backbone) | |
| `B` | Bước 2: ablation công thức huấn luyện trên backbone đã chọn | `CHOSEN_BACKBONE` chốt sau phiên A |
| `C` | Bước 2 (kết hợp) + Bước 3: phương pháp suy luận, độ trễ | `COMBOS` chốt sau phiên B |
| `D` | Bước 4: chung kết 3 seed, chạy test một lần mỗi seed, `eval.py score/grade` | cấu hình chốt sau phiên C |
| `all` | chạy tuần tự cả 4 phần (mặc định) | |

Các ô của phần khác bị bỏ qua. Output của phiên trước được gắn vào phiên sau (Kaggle *kernel sources*,
thư mục `/kaggle/input/...`). Bước 5 (`results.xlsx`, biểu đồ tổng hợp, báo cáo) chạy trên CPU bằng `make_results.py`.

**Quy tắc:** chọn mọi thứ trên **val**; test chỉ chạy ở phần `D`, đúng một lần cho mỗi seed.

In [1]:
# 0. Môi trường
import os, sys, glob, json, platform, subprocess, time, hashlib, zipfile, urllib.request
from pathlib import Path
try:
    import timm
    assert timm.__version__ == "1.0.30"
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.30"], check=True)
import numpy as np, pandas as pd, torch, torchvision, timm, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

CODE_DIR = Path.cwd()                         # thư mục code/ của bài nộp
REPO_DIR = CODE_DIR.parents[2]                # gốc repo (chứa eval.py)
sys.path.insert(0, str(CODE_DIR)); sys.path.insert(0, str(REPO_DIR))
ON_KAGGLE = Path("/kaggle/working").exists()
DATA = Path(os.environ.get("DATA_DIR", "/tmp/deepweeds" if ON_KAGGLE else "data"))
OUT = Path(os.environ.get("OUT_DIR", "/kaggle/working/out" if ON_KAGGLE else "out"))
STAGE = os.environ.get("LAB_STAGE", "all")
SMOKE = os.environ.get("LAB_SMOKE") == "1"   # chỉ để chạy thử pipeline trên CPU (vài bước, 64 ảnh val)
SMOKE_ARGS = ["--smoke"] if SMOKE else []
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def run_stage(s): return STAGE == "all" or s in STAGE.split("+")

env = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
       "cpu_count": os.cpu_count(), "stage": STAGE}
print(json.dumps(env, indent=1))
(OUT / f"env_{STAGE}.json").write_text(json.dumps(env, indent=1))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.3 MB/s eta 0:00:00
{
 "python": "3.13.15",
 "torch": "2.11.0+cu128",
 "torchvision": "0.26.0+cu128",
 "timm": "1.0.29",
 "numpy": "2.1.3",
 "pandas": "2.3.3",
 "cuda": "12.8",
 "cudnn": 91900,
 "gpus": [
  "Tesla T4",
  "Tesla T4"
 ],
 "cpu_count": 4,
 "stage": "C+D"
}


250

### Tải dữ liệu
Ảnh từ Zenodo (kiểm tra MD5 `b7b30f96d466fba86016aa5a26606e0f`), nhãn và file chia fold 0 nguyên bản từ GitHub
của tác giả. Ảnh được giải mã một lần thành mảng `uint8` (`cache.npy`, đọc bằng memmap) để DataLoader không phải
giải mã JPEG mỗi epoch; nội dung ảnh giống hệt đọc bằng PIL.

In [2]:
DATA.mkdir(parents=True, exist_ok=True); (DATA / "labels").mkdir(exist_ok=True)
zip_path = DATA / "images.zip"
if not zip_path.exists():
    urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", zip_path)
h = hashlib.md5()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
print("MD5 OK", h.hexdigest())
if not (DATA / "images").exists() or len(list((DATA / "images").glob("*.jpg"))) < 17509:
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA / "images")
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    p = DATA / "labels" / f"{name}.csv"
    if not p.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}.csv", p)
IMAGES_DIR, LABELS_DIR = DATA / "images", DATA / "labels"
print(len(list(IMAGES_DIR.glob("*.jpg"))), "ảnh;", sorted(os.listdir(LABELS_DIR)))

import dataset as D
train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
t0 = time.time()
D.build_cache(IMAGES_DIR, sorted(set(train_df.Filename) | set(val_df.Filename) | set(test_df.Filename)),
              DATA / "cache.npy")
print(f"cache: {time.time() - t0:.0f}s")

# Output của các phiên Kaggle trước (gắn qua kernel sources), để tìm run đã có
# đường dẫn gắn kernel source khác nhau giữa các phiên bản Kaggle (/kaggle/input/<slug>/ hoặc sâu hơn): tìm đệ quy
PREV = sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/out/runs", recursive=True)})
print("output phiên trước:", PREV)
def find_run(exp_id, seed=0):
    for root in [OUT] + PREV:
        p = root / "runs" / exp_id / f"seed{seed}"
        if (p / "summary.json").exists():
            return p
    raise FileNotFoundError(f"không thấy run {exp_id} seed{seed}")
def all_summaries():
    rows = {}
    for root in PREV + [OUT]:
        for f in glob.glob(str(root / "runs" / "*" / "seed*" / "summary.json")):
            s = json.loads(Path(f).read_text()); s["run_dir"] = str(Path(f).parent)
            rows[(s["exp_id"], s["seed"])] = s
    return pd.DataFrame(list(rows.values()))

MD5 OK b7b30f96d466fba86016aa5a26606e0f
17509 ảnh; ['labels.csv', 'test_subset0.csv', 'train_subset0.csv', 'val_subset0.csv']
cache: 28s
output phiên trước: [PosixPath('/kaggle/input/notebooks/viethwang3i/k4-day2-a-backbones/out'), PosixPath('/kaggle/input/notebooks/viethwang3i/k4-day2-b-ablations/out')]


## Bước 0 — Kiểm tra chia dữ liệu, EDA, kiểm tra pipeline

In [3]:
if run_stage("A"):
    split = D.check_split(train_df, val_df, test_df, IMAGES_DIR)
    (OUT / "split_check.json").write_text(json.dumps(split, indent=1))
    pc = pd.DataFrame(split["per_class"], index=D.CLASS_NAMES)
    pc["total"] = pc.sum(1); pc["paper_table1"] = [D.PAPER_TABLE1[c] for c in D.CLASS_NAMES]
    pc["diff_vs_paper"] = pc["total"] - pc["paper_table1"]
    pc.to_csv(OUT / "class_counts.csv")
    print(f"tỉ lệ lớp lớn nhất / nhỏ nhất (toàn bộ): {pc.total.max() / pc.total.min():.2f}; "
          f"train: {pc.train.max() / pc.train.min():.2f}")
    fig, ax = plt.subplots(figsize=(11, 4.5))
    x = np.arange(9); w = 0.27
    for i, s in enumerate(["train", "val", "test"]):
        b = ax.bar(x + (i - 1) * w, pc[s], w, label=f"{s} (n={split['n'][s]})")
        ax.bar_label(b, fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(D.CLASS_NAMES, rotation=20)
    ax.set_ylabel("số ảnh"); ax.set_title("DeepWeeds fold 0: số ảnh theo lớp và tập (Negatives ~52%)")
    ax.legend(); fig.tight_layout(); fig.savefig(FIG / "eda_class_distribution.png", dpi=120); plt.close(fig)
    display(ShowImage(str(FIG / "eda_class_distribution.png")))

    # >= 3 ảnh mỗi lớp (4 ảnh, lấy ngẫu nhiên có seed từ train)
    from PIL import Image
    rng = np.random.default_rng(0)
    fig, axes = plt.subplots(9, 4, figsize=(8, 18))
    for c in range(9):
        fs = rng.choice(train_df[train_df.Label == c].Filename.to_numpy(), 4, replace=False)
        for j, fn in enumerate(fs):
            axes[c, j].imshow(Image.open(IMAGES_DIR / fn)); axes[c, j].axis("off")
            axes[c, j].set_title(f"{D.CLASS_NAMES[c]}\n{fn}", fontsize=7)
    fig.tight_layout(); fig.savefig(FIG / "eda_samples.png", dpi=90); plt.close(fig)
    with Image.open(IMAGES_DIR / train_df.Filename.iloc[0]) as im:
        print("kích thước/kênh ảnh mẫu:", im.size, im.mode)

In [4]:
if run_stage("A"):
    # Kiểm tra pipeline (slide trang 59, GUIDE 1.3)
    import torch.nn.functional as F
    from model import build_model
    from losses import mix_batch
    from train import set_seed
    from run_experiments import BACKBONES
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(0)
    tf_eval = D.build_transforms(False, 224)
    vl = D.make_loader(val_df, IMAGES_DIR, tf_eval, 128, False, None, 2, 0, DATA / "cache.npy")
    batches = [b for _, b in zip(range(1 if SMOKE else 4), vl)]
    init_loss = {}
    for exp, name, _ in BACKBONES:   # 1) loss ban đầu của head mới ~ ln 9 = 2.197
        for head_init, std in (("timm mặc định", None), ("trunc_normal std 0.01 (dùng trong bài)", 0.01)):
            torch.manual_seed(0)
            m = build_model(name, True, 9, head_init_std=std).to(dev).eval()
            with torch.inference_mode():
                ls = [F.cross_entropy(m(x.to(dev)).float(), y.to(dev), reduction="sum").item() for x, y, _ in batches]
            init_loss.setdefault(name, {})[head_init] = sum(ls) / sum(len(b[1]) for b in batches)
            del m
    print(f"-ln(1/9) = {np.log(9):.4f}")
    print(pd.DataFrame(init_loss).T.round(4).to_string())

    # 2) overfit một batch nhỏ (16 ảnh train, không augmentation) tới loss ~ 0
    set_seed(0)
    small = D.make_loader(train_df.iloc[:16], IMAGES_DIR, tf_eval, 16, False, None, 0, 0, DATA / "cache.npy")
    xb, yb, _ = next(iter(small)); xb, yb = xb.to(dev), yb.to(dev)
    m = build_model("resnet50.a1_in1k", True, 9).to(dev).train()
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0)
    hist = []
    for step in range(11 if SMOKE else 101):
        loss = F.cross_entropy(m(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 10 == 0:
            hist.append((step, loss.item()))
    m.eval()
    with torch.inference_mode():
        acc = (m(xb).argmax(1) == yb).float().mean().item()
    print("overfit 16 ảnh (resnet50, AdamW 3e-4):", [(s, round(l, 4)) for s, l in hist], f"| acc eval {acc:.3f}")
    (OUT / "pipeline_checks.json").write_text(json.dumps({"ln9": float(np.log(9)), "initial_loss": init_loss,
                                                          "overfit_loss": hist, "overfit_acc_eval": acc}, indent=1))
    del m, opt

    # 3) ảnh sau augmentation (đã giải chuẩn hoá) + nhãn; CutMix trộn cả ảnh và nhãn
    ds = D.DeepWeedsDataset(train_df, IMAGES_DIR, None, cache=DATA / "cache.npy")
    idx = [int(train_df.index[train_df.Label == c][0]) for c in (0, 7, 8)]
    augs = ["basic", "color", "trivial", "randaug"]
    fig, axes = plt.subplots(len(idx), 1 + 2 * len(augs), figsize=(20, 7))
    torch.manual_seed(0)
    for r, i in enumerate(idx):
        img, lab, fn = ds[i]
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"gốc: {D.CLASS_NAMES[lab]}", fontsize=8)
        for a, aug in enumerate(augs):
            tf = D.build_transforms(True, 224, aug)
            for k in range(2):
                ax = axes[r, 1 + 2 * a + k]
                ax.imshow(D.denormalize(tf(img)).permute(1, 2, 0).numpy())
                ax.set_title(f"{aug} #{k}: {D.CLASS_NAMES[lab]}", fontsize=8)
    for ax in axes.ravel(): ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG / "aug_samples.png", dpi=80); plt.close(fig)
    display(ShowImage(str(FIG / "aug_samples.png")))
    tf = D.build_transforms(True, 224, "basic")
    xs = torch.stack([tf(ds[i][0]) for i in range(8)]); ys = torch.tensor([ds[i][1] for i in range(8)])
    xm, (ya, yb2, lam) = mix_batch(xs, ys, 1.0, "cutmix", rng=np.random.default_rng(1))
    fig, axes = plt.subplots(1, 8, figsize=(20, 3.2))
    for j in range(8):
        axes[j].imshow(D.denormalize(xm[j]).permute(1, 2, 0).numpy()); axes[j].axis("off")
        axes[j].set_title(f"{lam:.2f}·{D.CLASS_NAMES[ya[j]]}\n+{1 - lam:.2f}·{D.CLASS_NAMES[yb2[j]]}", fontsize=8)
    fig.suptitle(f"CutMix (alpha=1): lam theo diện tích thật = {lam:.3f}")
    fig.tight_layout(); fig.savefig(FIG / "cutmix_samples.png", dpi=80); plt.close(fig)
    display(ShowImage(str(FIG / "cutmix_samples.png")))

In [5]:
if run_stage("A"):
    # Kiểm tra tự viết cho phần dễ sai (focal gamma=0 == CE, CutMix, gộp BN, LR schedule...)
    r = subprocess.run([sys.executable, "-m", "unittest", "test_code", "-v"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-3000:], r.stderr[-6000:])

## Bước 1 — So sánh backbone (7 backbone, công thức nền `T00`, seed 0)
Công thức nền (GUIDE 1.4): ImageNet-1k pretrained + head mới 9 lớp, tinh chỉnh toàn bộ; train `RandomResizedCrop(224)` +
lật ngang; val/test `CenterCrop(224)` từ ảnh 256; AdamW, LR backbone 1e-4 / head 1e-3, weight decay 0.05 (không áp
dụng cho norm/bias), warmup 1 epoch + cosine theo bước, CE, batch 64, 12 epoch, AMP FP16, chọn epoch có macro-F1 val
cao nhất. Hai job chạy song song trên 2 GPU T4.

In [6]:
if run_stage("A"):
    r = subprocess.run([sys.executable, "run_experiments.py", "--stage", "backbones", "--data", str(DATA),
                        "--out", str(OUT), *SMOKE_ARGS], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-30000:]); print(r.stderr[-5000:])
    assert r.returncode == 0
    s = all_summaries()
    cols = ["exp_id", "backbone", "weight_tag", "params_m", "gmacs", "val_macro_f1", "val_top1", "best_epoch",
            "train_time_per_epoch_s"]
    print(s[s.exp_id.str.startswith("B")][cols].sort_values("exp_id").to_string(index=False))

## Bước 2 — Công thức huấn luyện (backbone `deit_small_patch16_224.fb_in1k`, seed 0)

**Lựa chọn backbone (sau phiên A, chỉ dựa trên val):** Trên val (1 seed), ba backbone dẫn đầu sát nhau: DeiT-S (B04) macro-F1 0.9556, ConvNeXt-T (B03) 0.9507, Swin-T (B05) 0.9504; chênh lệch ≤ 0.005 nên coi là không phân biệt được với 1 seed. Giữa ba mạng này DeiT-S rẻ nhất: ít tham số nhất (21.7M), thời gian train/epoch thấp nhất (31 s so với 52 s và 60 s), nên chọn DeiT-S để chạy nhiều ablation hơn trong ngân sách GPU. Hai mạng nhẹ (EfficientNet-B0 0.8500, MobileNetV3-L 0.7601) và ResNet/ResNeXt trọng số a1 (0.8019/0.7050) kém rõ rệt.

Mỗi `T0x` khác `T00` đúng một yếu tố (trục A khởi tạo, B augmentation, C loss, D sampler, F EMA).
`T00` chạy 3 seed để ước lượng nhiễu; 3 run này cũng là mốc (`T00` + `I00`) ở Bước 4.

In [7]:
CHOSEN_BACKBONE = "deit_small_patch16_224.fb_in1k"
if run_stage("B"):
    # kèm B08 (ResNet-50 trọng số torchvision, thêm sau phiên A - lý do ở report mục 3) trong cùng hàng đợi
    r = subprocess.run([sys.executable, "run_experiments.py", "--stage", "ablations", "--backbone", CHOSEN_BACKBONE,
                        "--set", json.dumps({"with_backbones": ["B08"]}),
                        "--data", str(DATA), "--out", str(OUT), *SMOKE_ARGS], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-30000:]); print(r.stderr[-5000:])
    assert r.returncode == 0

## Bước 2 (tiếp) — Kết hợp các yếu tố tốt, và Bước 3 — Suy luận

**Kết hợp (chốt sau phiên B, chỉ dựa trên val):** T00 (3 seed) có macro-F1 val 0.9505 ± 0.0026. Chỉ T04 TrivialAugment vượt rõ nhiễu: 0.9629 (Δ +0.0124 ≈ 4.8 std). Mixup T06 (+0.0059), color jitter T03 (+0.0040), label smoothing T07 (+0.0029) nằm trong khoảng 1–2.3 std nên chỉ là tín hiệu yếu (1 seed); các yếu tố còn lại không phân biệt được hoặc kém hơn (frozen −0.207, scratch −0.304). Do trễ hạn nên phiên C và D được gộp thành một phiên: công thức chung kết F01 = T00 + TrivialAugment được chốt NGAY từ kết quả val của phiên B (trước khi thấy kết hợp), và kết hợp T12 = TrivialAugment + Mixup(α=0.2) + label smoothing 0.1 chạy song song chỉ để phân tích cộng dồn/triệt tiêu, không dùng để chọn F01.

In [8]:
COMBOS = {"T12_trivial_mixup_ls": {"aug": "trivial", "mix": "mixup", "mix_alpha": 0.2, "loss": "ls", "label_smoothing": 0.1}, "F01_seed0_final": {"aug": "trivial", "seed": 0}, "F01_seed1_final": {"aug": "trivial", "seed": 1}, "F01_seed2_final": {"aug": "trivial", "seed": 2}}
def _exists(key, ch):
    try:
        return find_run(key.split("_")[0], ch.get("seed", 0)) is not None
    except FileNotFoundError:
        return False
if run_stage("C"):
    todo = {k: v for k, v in COMBOS.items() if not _exists(k, v)}   # run đã có (phiên trước) thì không train lại
    print("đã có:", sorted(set(COMBOS) - set(todo)), "| cần train:", sorted(todo))
    if todo:
        r = subprocess.run([sys.executable, "run_experiments.py", "--stage", "combo", "--backbone", CHOSEN_BACKBONE,
                            "--set", json.dumps(todo), "--data", str(DATA), "--out", str(OUT), *SMOKE_ARGS],
                           capture_output=True, text=True, encoding="utf-8", errors="replace")
        print(r.stdout[-20000:]); print(r.stderr[-5000:])
        assert r.returncode == 0
    s = all_summaries()
    t = s[s.exp_id.str.startswith("T")].sort_values("val_macro_f1", ascending=False)
    print(t[["exp_id", "seed", "val_macro_f1", "val_top1", "best_epoch"]].to_string(index=False))
    best = s[(s.exp_id == "T04") & (s.seed == 0)].iloc[0]   # model của công thức chung kết
    print("model cho Bước 3:", best.exp_id, "seed", best.seed, f"val F1 {best.val_macro_f1:.4f}")
    from run_experiments import BACKBONES
    args = [sys.executable, "inference_experiments.py", "--data", str(DATA), "--out", str(OUT),
            "--best-run", best.run_dir,
            "--t00-runs", *[str(find_run("T00", k)) for k in (0, 1, 2)],
            "--ensemble-runs", *[str(find_run(e)) for e in ["B03", "B05"]],
            "--ema-run", str(find_run("T11")), "--bn-run", str(find_run("B08")),
            "--latency-backbones", *[b for _, b, _ in BACKBONES]] + (["--limit", "64"] if SMOKE else [])
    r = subprocess.run(args, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-30000:]); print(r.stderr[-5000:])
    assert r.returncode == 0

đã có: [] | cần train: ['F01_seed0_final', 'F01_seed1_final', 'F01_seed2_final', 'T12_trivial_mixup_ls']
cache ảnh: /tmp/deepweeds/cache.npy (0s)
stage combo: 4 job trên GPU 0,1
  ... 0/4 job xong sau 0.5 phút
  ... 0/4 job xong sau 1.0 phút
  ... 0/4 job xong sau 1.5 phút
  ... 0/4 job xong sau 2.0 phút
  ... 0/4 job xong sau 2.5 phút
  ... 0/4 job xong sau 3.0 phút
  ... 0/4 job xong sau 3.5 phút
  ... 0/4 job xong sau 4.0 phút
  ... 0/4 job xong sau 4.5 phút
  ... 0/4 job xong sau 5.0 phút
  ... 0/4 job xong sau 5.5 phút
  ... 0/4 job xong sau 6.0 phút
  ... 0/4 job xong sau 6.5 phút
  ... 0/4 job xong sau 7.0 phút
  ... 0/4 job xong sau 7.5 phút
  ... 0/4 job xong sau 8.0 phút
  ... 1/4 job xong sau 8.5 phút
  ... 1/4 job xong sau 9.0 phút
  ... 2/4 job xong sau 9.5 phút
  ... 2/4 job xong sau 10.0 phút
  ... 2/4 job xong sau 10.5 phút
  ... 2/4 job xong sau 11.0 phút
  ... 2/4 job xong sau 11.5 phút
  ... 2/4 job xong sau 12.0 phút
  ... 2/4 job xong sau 12.5 phút
  ... 2/4 job xo

## Bước 4 — Chung kết: 3 seed, test một lần mỗi seed

**Cấu hình chốt (sau phiên C, chỉ dựa trên val):** Backbone DeiT-S (phiên A), công thức T00 + TrivialAugment (T04, phiên B, thắng rõ trên val). Phương pháp suy luận chọn bằng quy tắc viết sẵn trước khi chạy test, chỉ dùng val của model T04 seed 0: TTA lật ngang (I01, gộp logit) nếu macro-F1 val cao hơn 1-view (I00), ngược lại 1-view; luôn kèm temperature scaling với T khớp trên val.

`F01` = `deit_small_patch16_224.fb_in1k` + `{"aug": "trivial"}` + suy luận
`auto` (gộp logit) + temperature scaling (T khớp trên val).
`R01` = cùng model, 1 view (cấu hình thời gian thực). Mốc = `T00` (3 seed của Bước 2) + `I00`.

In [9]:
FINAL = {"aug": "trivial"}
if run_stage("D"):
  try:
    have_f01 = all(find_run("F01", k) for k in (0, 1, 2))   # F01 đã train chung hàng đợi ở ô Bước 3
  except FileNotFoundError:
    have_f01 = False
  if not have_f01:
    r = subprocess.run([sys.executable, "run_experiments.py", "--stage", "final", "--backbone", CHOSEN_BACKBONE,
                        "--set", json.dumps({**FINAL, "seeds": [0, 1, 2]}), "--data", str(DATA), "--out", str(OUT), *SMOKE_ARGS],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-20000:]); print(r.stderr[-5000:])
    assert r.returncode == 0
  if True:
    # Quy tắc chọn suy luận, chốt TRƯỚC khi chạy test và chỉ dùng val (model T04 seed 0, Bước 3):
    # dùng I01 (TTA lật, gộp logit) nếu macro-F1 val cao hơn I00, ngược lại I00 (rẻ hơn).
    METHOD = "auto"
    if METHOD == "auto":
        rows = json.loads((OUT / "inference" / "inference_val.json").read_text())["rows"]
        f1 = {r_["method"]: r_["val_macro_f1"] for r_ in rows if r_["exp_id"] in ("I00", "I01")}
        METHOD = "I01_hflip" if f1["I01_hflip"] > f1["I00_1view"] else "I00_1view"
        print("macro-F1 val:", f1, "-> phương pháp chung kết:", METHOD)
    args = [sys.executable, "final_eval.py", "--data", str(DATA), "--out", str(OUT),
            "--final-runs", *[str(find_run("F01", k)) for k in (0, 1, 2)],
            "--baseline-runs", *[str(find_run("T00", k)) for k in (0, 1, 2)],
            "--method", METHOD, "--space", "logit", "--ts",
            "--rt-method", "I00_1view"] + (["--limit", "64"] if SMOKE else [])
    r = subprocess.run(args, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-20000:]); print(r.stderr[-5000:])
    assert r.returncode == 0

macro-F1 val: {'I00_1view': 0.9628741250563141, 'I01_hflip': 0.9640102598071232} -> phương pháp chung kết: I01_hflip
F01 seed0: T=1.0410 T_rt=1.0573 -> đã ghi predictions
F01 seed1: T=0.9948 T_rt=1.0178 -> đã ghi predictions
F01 seed2: T=1.0881 T_rt=1.0906 -> đã ghi predictions
T00 seed0: đã ghi predictions
T00 seed1: đã ghi predictions
T00 seed2: đã ghi predictions
F01 (chung kết)        I01_hflip    b1   fp32  p50 10.66 p95 11.02 p99 11.74 ms
F01 (chung kết)        I01_hflip    b1   amp   p50 12.69 p95 14.44 p99 18.84 ms
F01 (chung kết)        I01_hflip    b1   fp16  p50 9.33 p95 10.29 p99 11.51 ms
F01 (chung kết)        I01_hflip    b32  fp32  p50 227.89 p95 235.62 p99 237.06 ms
F01 (chung kết)        I01_hflip    b32  amp   p50 59.29 p95 59.84 p99 59.99 ms
F01 (chung kết)        I01_hflip    b32  fp16  p50 51.76 p95 52.47 p99 52.51 ms
R01 (thời gian thực)   I00_1view    b1   fp32  p50 5.32 p95 5.46 p99 6.53 ms
R01 (thời gian thực)   I00_1view    b1   amp   p50 6.28 p95 6.73 p99 7.1

In [10]:
if run_stage("D"):
    P, L = OUT / "predictions", LABELS_DIR
    common = ["--test-csv", str(L / "test_subset0.csv"), "--labels", str(L / "labels.csv"), "--out", str(OUT / "eval_out")]
    for tag in ("F01", "F01_uncal", "R01", "T00"):
        r = subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "score", "--pred", str(P / f"{tag}_seed*_test.csv"),
                            "--tag", tag, *common], capture_output=True, text=True, encoding="utf-8", errors="replace")
        print(r.stdout, r.stderr)
    rec = json.loads((OUT / "final" / "final_record.json").read_text())
    p95 = [x["p95"] for x in rec["latency"] if x["method"] == "I00_1view" and x["batch"] == 1 and x["dtype"] == "fp16"]
    lat_args = ["--latency-p95-ms", f"{p95[0]:.2f}", "--latency-method", "proper"] if p95 else []
    print("p95 batch-1 của R01 (FP16):", p95)
    r = subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "grade",
                        "--final", str(P / "F01_seed*_test.csv"), "--baseline", str(P / "T00_seed*_test.csv"),
                        "--uncal", str(P / "F01_uncal_seed*_test.csv"), "--final-val", str(P / "F01_seed*_val.csv"),
                        "--val-csv", str(L / "val_subset0.csv"), *lat_args, *common], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout, r.stderr)

### F01 (3 seed: [0, 1, 2]; 3507 ảnh)

| Chỉ số | mean ± std |
|---|---|
| top-1 accuracy | 0.9693 ± 0.0004 |
| macro-F1 | 0.9605 ± 0.0018 |
| balanced accuracy | 0.9590 ± 0.0022 |
| ECE (15 bin) | 0.0079 ± 0.0011 |
| NLL | 0.0905 ± 0.0020 |

| Lớp | Precision | Recall | F1 | Số ảnh |
|---|---|---|---|---|
| Chinee apple | 0.961 ± 0.019 | 0.891 ± 0.013 | 0.924 ± 0.006 | 226 |
| Lantana | 0.962 ± 0.017 | 0.981 ± 0.005 | 0.971 ± 0.007 | 213 |
| Parkinsonia | 0.978 ± 0.010 | 0.992 ± 0.006 | 0.985 ± 0.005 | 207 |
| Parthenium | 0.980 ± 0.010 | 0.954 ± 0.006 | 0.967 ± 0.007 | 205 |
| Prickly acacia | 0.926 ± 0.010 | 0.953 ± 0.012 | 0.939 ± 0.004 | 213 |
| Rubber vine | 0.977 ± 0.010 | 0.960 ± 0.005 | 0.968 ± 0.004 | 202 |
| Siam weed | 0.968 ± 0.008 | 0.983 ± 0.005 | 0.975 ± 0.003 | 215 |
| Snake weed | 0.933 ± 0.013 | 0.935 ± 0.010 | 0.934 ± 0.002 | 204 |
| Negative | 0.978 ± 0.007 | 0.982 ± 0.003 | 0.980 ± 0.002 | 1822 |

Đã lưu kết quả vào /kaggle/working/out/eval_out/F01_*
 
### F01_unc

## Bước 5 — Sản phẩm
`results.xlsx`, biểu đồ tổng hợp (đánh đổi độ chính xác–độ trễ, ma trận nhầm lẫn, ảnh bị đoán sai) và số liệu cho
báo cáo được tạo trên CPU từ output của 4 phiên bằng `python make_results.py` (xem README của bài nộp).